### Loading Bhasha-Abhijnaanam Datasets

In [33]:
import os, zipfile, requests

urls = {
    "native": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/native_script_train_valid_data.zip",
    "roman": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/roman_script_train_valid_data.zip",
    "native-roman": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/parallel_romanized_train_data.zip",
    "abhijnaanam": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/bhasha-abhijnaanam_test_set.zip"
}

os.makedirs("ba_training", exist_ok=True)

for name, url in urls.items():
    zip_path = f"ba_training/{name}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {name} dataset...")
        r = requests.get(url)
        with open(zip_path, "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(f"ba_training/")
        os.remove(zip_path)
        print("Zip file removed.")
os.listdir("ba_training/")

Zip file removed.
Zip file removed.
Zip file removed.
Zip file removed.


['parallel_romanized_train_data.json',
 'Native_script_data',
 'Roman_script_data',
 'bhasha-abhijnaanam.json']

In [34]:
import pandas as pd

def load_and_sample(path, frac=1, seed=42):
    """Load the text file, split label and text, and sample a portion of data."""
    data = []
    with open(path, 'r') as f:
        for line in f:
            parts = line.strip().split(maxsplit=1)  
            if len(parts) == 2:
                label, text = parts
                label = label.replace("__label__","")
                data.append([label, text])  

    df = pd.DataFrame(data, columns=["label", "text"])
    return df

roman_train = load_and_sample("ba_training/Roman_script_data/train_combine.txt")
roman_valid = load_and_sample("ba_training/Roman_script_data/valid_combine.txt")
roman = pd.concat([roman_train, roman_valid], axis=0, ignore_index=True)
native_train = load_and_sample("ba_training/Native_script_data/train_combine.txt")
native_valid = load_and_sample("ba_training/Native_script_data/valid_combine.txt")
native = pd.concat([native_train, native_valid], axis=0, ignore_index=True)

In [35]:
import json

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)   # one JSON object
    data = obj["data"]
    df = pd.DataFrame([{
        "id": d["unique_identifier"],
        "native": d["native sentence"],
        "roman": d["romanized sentence"],
        "label": d["language"],
        "script": d["script"],
        "source": d["source"]
    } for d in data])

    return df

# usage
benchmark = load_json("ba_training/bhasha-abhijnaanam.json")
roman_native = load_json("ba_training/parallel_romanized_train_data.json")

In [36]:
benchmark_native = benchmark[['native', 'label']].rename(columns={'native': 'text'})
benchmark_roman = benchmark[['roman', 'label']].rename(columns={'roman': 'text'})
benchmark_roman = benchmark_roman[benchmark_roman['text'] != '']
benchmark = pd.concat([benchmark_native, benchmark_roman], axis=0, ignore_index=True)
benchmark.to_csv('bhasha-abhijnaanam.csv', index=False)

In [37]:
native = native[native['label'].isin(['Dogri', 'Santali', 'Manipuri_Beng', 'Manipuri_Mei'])]
native = native.rename(columns = {'text': 'native'})
native.label.value_counts()     # adding these classes to solve class imbalance and add missing classes

label
Manipuri_Beng    100997
Manipuri_Mei     100500
Santali          100345
Dogri            100120
Name: count, dtype: int64

### Using Custom transliteration (Aksharmukha + mapping) for creating roman scripts

In [38]:
from aksharamukha.transliterate import process
import re

In [39]:
santhali_map = {
    '᱐': '0', '᱑': '1', '᱒': '2', '᱓': '3', '᱔': '4', '᱕': '5', '᱖': '6', '᱗': '7', '᱘': '8', '᱙': '9',
    'ᱚ': 'a', 'ᱛ': 'b', 'ᱜ': 'c', 'ᱝ': 'd', 'ᱞ': 'e', 'ᱟ': 'f', 'ᱠ': 'g', 'ᱡ': 'h',
    'ᱢ': 'i', 'ᱣ': 'j', 'ᱤ': 'k', 'ᱥ': 'l', 'ᱦ': 'm', 'ᱧ': 'n', 'ᱨ': 'o', 'ᱩ': 'p',
    'ᱪ': 'q', 'ᱫ': 'r', 'ᱬ': 's', 'ᱭ': 't', 'ᱮ': 'u', 'ᱯ': 'v', 'ᱰ': 'w', 'ᱱ': 'x',
    'ᱲ': 'y', 'ᱳ': 'z', 'ᱴ': 'ṭ', 'ᱵ': 'ḍ', 'ᱶ': 'ṅ', 'ᱷ': 'ṭh', 'ᱸ': 'ḍh', 'ᱹ': 'ñ',
    'ᱺ': 'ŋ', 'ᱻ': 'ś', 'ᱼ': 'ṣ', 'ᱽ': 'ḷ'
}

language_to_script = {
    'Dogri': 'Devanagari',
    'Manipuri_Beng': 'Bengali',
    'Manipuri_Mei': 'MeeteiMayek'
}
custom_mappings = {
    'Santali': santhali_map,
}

In [40]:
def transliterate_row(row):
    lang = row['label']
    sentence = str(row['native'])
    
    # If custom mapping language
    if lang in custom_mappings:
        mapping = custom_mappings[lang]
        return ''.join([mapping.get(c, c) for c in sentence])
    
    # Use Aksharamukha for all other languages
    source_script = language_to_script.get(lang)
    target_script = 'IAST'  # Roman
    if not source_script:
        return '[Unknown language]'
    try:
        return process(source_script, target_script, sentence)
    except Exception as e:
        return f"[Error: {e}]"

In [41]:
native['roman'] = native.apply(transliterate_row, axis=1)
native

,label,native,roman
3,Manipuri_Mei,ꯍꯤꯕꯤ ꯅꯠꯇ꯭ꯔꯒ ꯍꯦꯕꯦ ꯑꯁꯤ ꯒ꯭ꯔꯤꯛꯀꯤ ꯃꯤ ꯂꯥꯢ ꯇꯤꯟꯒꯤ ꯋꯥꯔꯤ...,hibi nattraga hebe asi grikki mi lāy tingi vār...
9,Manipuri_Beng,মতম অদুদা লৈরম্বা য়ুম্নাক ৩৬১ অদুদা চপ মান্নন...,matama adudā lairambā ẏumnāka 361 adudā capa m...
14,Manipuri_Beng,খূদম ওইনা নুপাগী সেক্স ওর্গানদগী অঙাং ওইহনবদা ...,khūdama oinā nupāgī seksa orgānadagī aṅāṃ oiha...
30,Manipuri_Mei,ꯀꯣꯚꯦꯂꯦꯟꯠ ꯂꯤꯄꯨꯟ ꯅꯠꯇ꯭ꯔꯒ ꯃꯣꯂꯦꯀꯨꯂꯔ ꯂꯤꯄꯨꯟ ( ꯏꯪꯂꯤꯁ :...,kobhelent lipun nattraga molekulara lipun ( iṃ...
37,Dogri,जे दक्खना च,je dakkhanā ca
...,...,...,...
2710676,Manipuri_Beng,1965কী মার্চ 18দা মহাক্না স্পেসক্রাফকী মপান্দা...,1965kī mārca 18dā mahāknā spesakrāphakī mapānd...
2710714,Manipuri_Mei,ꯏꯪ ꯱꯹꯹꯹ ꯗ ꯕꯣꯕꯤꯅ ꯃꯍꯥꯛꯀꯤ ꯃꯌꯥꯝꯕ ꯁꯅꯤ ꯗꯤꯑꯣꯜꯅ ꯂꯝꯖꯤꯡ...,iṃ 1999 da bobina mahākki mayāmba sani diolna...
2710805,Manipuri_Mei,ꯚꯥꯔꯇꯅꯥ ꯏꯪ ꯲꯰꯰꯱ꯗꯥ ꯈꯥ ꯑꯐ꯭ꯔꯤꯀꯥꯗꯥ ꯆꯠꯈꯤꯕꯥ ꯈꯣꯡꯆꯠ ꯑꯗꯨ...,bhāratanā iṃ 2001dā khā aphrikādā catkhibā kho...
2710849,Manipuri_Mei,ꯃꯥꯂꯦꯝꯒꯤ ꯊꯥꯛꯇ ꯃꯃꯤꯡ ꯆꯠꯂꯕ ꯃꯤꯇꯝ ꯁꯥꯕ ꯃꯤꯑꯣꯏ ꯁꯨꯁꯤꯜ ꯁꯈ...,mālemgi thākta mamiṅ catlaba mitam sāba mioi s...


In [42]:
native.label.value_counts()

label
Manipuri_Beng    100997
Manipuri_Mei     100500
Santali          100345
Dogri            100120
Name: count, dtype: int64

In [43]:
manipuri_beng = native[native['label'] == 'Manipuri_Beng']
manipuri_mei = native[native['label'] == 'Manipuri_Mei']

n_each = 100_500 // 2  # 50,250 from each script

manipuri_beng_sampled = manipuri_beng.sample(n=n_each, random_state=42)
manipuri_mei_sampled = manipuri_mei.sample(n=n_each, random_state=42)
manipuri_balanced = pd.concat([manipuri_beng_sampled, manipuri_mei_sampled], ignore_index=True)
manipuri_balanced['label'] = 'Manipuri'
native_rest = native[~native['label'].isin(['Manipuri_Beng', 'Manipuri_Mei'])]
native_final = pd.concat([native_rest, manipuri_balanced], ignore_index=True)
native_final.label.value_counts()

label
Manipuri    100500
Santali     100345
Dogri       100120
Name: count, dtype: int64

### Merging Parallel Dataset with Native Dataset to solve Class Imbalance 

In [44]:
merged = pd.concat([native_final, roman_native], ignore_index=True)
merged.label.value_counts()

label
Telugu       299033
Gujarati     298984
Bangla       298926
Malayalam    298812
Marathi      298795
Hindi        298612
Oriya        296294
Tamil        292926
Kannada      289949
Assamese     279066
Punjabi      235848
Nepali       234596
Sanskrit     201462
Maithili     156921
Sindhi       150751
Manipuri     131028
Bodo         114102
Konkani      110001
Urdu         105704
Kashmiri     105654
Santali      100345
Dogri        100120
Name: count, dtype: int64

### Adding random english sentence(from Samanatar) for creating triplets

In [45]:
from datasets import load_dataset
import random

langs = ["hi", "bn", "ta", "ml", "te", "gu", "mr", "pa", "or", "kn", "as"]
english_sentences = []

for lang in langs:
    try:
        print(f"Loading {lang} split from Samanantar...")
        ds = load_dataset("ai4bharat/samanantar", lang, split="train[:2000000]")

        # Filter sentences that are strings and have between 10 and 50 words
        filtered = [
            row["src"] for row in ds
            if isinstance(row["src"], str)
            and 10 <= len(row["src"].split()) 
        ]

        english_sentences.extend(filtered)
        print(f"Loaded {len(filtered):,} valid English sentences for {lang}")

    except Exception as e:
        print(f"Skipping {lang} due to error: {e}")

print(f"Total English sentences collected: {len(english_sentences):,}")

# Sanity check
if not english_sentences:
    raise RuntimeError("No English sentences collected! Check dataset structure or filtering criteria.")

# Randomly assign one English sentence to each row
merged["english"] = random.sample(
    english_sentences, k=len(merged)
)

print("Added 'english_random_sentence' column from Samanantar dataset.")

Loading hi split from Samanantar...
Loaded 1,331,605 valid English sentences for hi
Loading bn split from Samanantar...
Loaded 830,087 valid English sentences for bn
Loading ta split from Samanantar...
Loaded 831,200 valid English sentences for ta
Loading ml split from Samanantar...
Loaded 766,793 valid English sentences for ml
Loading te split from Samanantar...
Loaded 692,563 valid English sentences for te
Loading gu split from Samanantar...
Loaded 792,640 valid English sentences for gu
Loading mr split from Samanantar...
Loaded 875,439 valid English sentences for mr
Loading pa split from Samanantar...
Loaded 1,074,036 valid English sentences for pa
Loading or split from Samanantar...
Loaded 437,213 valid English sentences for or
Loading kn split from Samanantar...
Loaded 650,923 valid English sentences for kn
Loading as split from Samanantar...
Loaded 66,903 valid English sentences for as
Total English sentences collected: 8,349,402
Added 'english_random_sentence' column from Samana

In [46]:
merged.drop(columns=['id','script','source'], inplace=True)

In [47]:
merged.label.value_counts()

label
Telugu       299033
Gujarati     298984
Bangla       298926
Malayalam    298812
Marathi      298795
Hindi        298612
Oriya        296294
Tamil        292926
Kannada      289949
Assamese     279066
Punjabi      235848
Nepali       234596
Sanskrit     201462
Maithili     156921
Sindhi       150751
Manipuri     131028
Bodo         114102
Konkani      110001
Urdu         105704
Kashmiri     105654
Santali      100345
Dogri        100120
Name: count, dtype: int64

### Phase1 Dataset Creation

In [48]:
sampled_dfs = []

for lang, group in merged.groupby("label"): 
    sampled = group.sample(n=100120, random_state=42)
    sampled_dfs.append(sampled)

balanced_triplet = pd.concat(sampled_dfs, ignore_index=True)
print("Samples per language (after undersampling):")
print(balanced_triplet["label"].value_counts())

# Calculate size in bytes → MB / GB
df_size_bytes = balanced_triplet.memory_usage(deep=True).sum()
df_size_mb = df_size_bytes / (1024 ** 2)
df_size_gb = df_size_bytes / (1024 ** 3)

print(f"\nTotal samples: {len(balanced_triplet):,}")
print(f"Languages included: {balanced_triplet['label'].nunique()}")
print(f"DataFrame size: {df_size_mb:.2f} MB ({df_size_gb:.3f} GB)")

Samples per language (after undersampling):
label
Assamese     100120
Bangla       100120
Bodo         100120
Dogri        100120
Gujarati     100120
Hindi        100120
Kannada      100120
Kashmiri     100120
Konkani      100120
Maithili     100120
Malayalam    100120
Manipuri     100120
Marathi      100120
Nepali       100120
Oriya        100120
Punjabi      100120
Sanskrit     100120
Santali      100120
Sindhi       100120
Tamil        100120
Telugu       100120
Urdu         100120
Name: count, dtype: int64

Total samples: 2,202,640
Languages included: 22
DataFrame size: 1405.49 MB (1.373 GB)


In [49]:
phase1 = balanced_triplet.drop('label', axis=1)
phase1

,native,roman,english
0,আনফালে আৰক্ষীৰ অভিযোগ একাংশ লোকে পৰিকল্পিত ভাৱ...,unfale aarokhyir obhijug ekangxo luke porikolp...,"Banks, after that date, decided to pass on the..."
1,"যেনেদৰে শাৰিৰীক দুখ , কষ্ট সহ্য কৰিবলগীয়া হয়...","jenedore xaririk dukh , kosto xohyo koribologi...","Given the potential for tension, police have p..."
2,প্ৰবায়োটিকছ কেপছুলো বজাৰত পোৱা যায় ।,probayutiks capsulu bojarot puwa yaay .,"""Moreover the word of Yahweh came to me, sayin..."
3,"জানিব পৰা মতে , মটৰচাইকেল আৰোহীজন বকো ২নং ছেখা...","janibo poraa mote , motorsaikel aaruhijon boku...",This afternoon Fiji's military regime expelled...
4,অন্যথা অনাগত দিনত ইয়াতকৈ জংগী আন্দোলনৰ কাৰ্যস...,onyotha onagoto dinot iyatkoi jongi andulonor ...,"""""""The issue of Jammu and Kashmir comes up in ..."
...,...,...,...
2202635,آپ سوچ رہیں ہوں گیں کہ عمران خان ابھی تک کیا ک...,aap soch rahein hoan gain kahh amraan khaan ub...,"There is some evidence that the anion, which o..."
2202636,انگلینڈ جرنل آف میڈیسن میں شائع ہونے ہونے وال...,england journal aff medison mein shaye honay h...,Twitter and other social media services like F...
2202637,خزاں میں تم کو خرید لیں گے,khazaan mein tamm kuu khareed lain gay,JNU protest was being organized outside the AB...
2202638,سردار گل محمد خان جوگیزئی ( پیدائش : ، ضلع لور...,sardaar gull mohammad khaan jogeezi ( paidaish...,Teams of the State Disaster Response Force (SD...


In [50]:
phase1.to_csv('phase1.csv', index=False)

### Phase2 Dataset Creation

In [51]:
sampled_dfs = []

for lang, group in merged.groupby("label"): 
    sampled = group.sample(n=21454, random_state=24)
    sampled_dfs.append(sampled)

balanced = pd.concat(sampled_dfs, ignore_index=True)
print("Samples per language (after undersampling):")
print(balanced["label"].value_counts())

Samples per language (after undersampling):
label
Assamese     21454
Bangla       21454
Bodo         21454
Dogri        21454
Gujarati     21454
Hindi        21454
Kannada      21454
Kashmiri     21454
Konkani      21454
Maithili     21454
Malayalam    21454
Manipuri     21454
Marathi      21454
Nepali       21454
Oriya        21454
Punjabi      21454
Sanskrit     21454
Santali      21454
Sindhi       21454
Tamil        21454
Telugu       21454
Urdu         21454
Name: count, dtype: int64


In [52]:
phase2_native = balanced[['native','label']].rename(columns={'native':'text'})
phase2_roman = balanced[['roman','label']].rename(columns={'roman':'text'})
phase2 = pd.concat([phase2_native, phase2_roman], axis=0, ignore_index=True)
phase2

,text,label
0,গুৱাহাটী উচ্চ ন্যায়ালয় অসমৰ সৰ্বোচ্চ ন্যায়া...,Assamese
1,যিবোৰ কাৰখানাৰ মেচিন আৰু তালৈ অহা যোৱা শ শ ডাম...,Assamese
2,ই অতি বিশাল ৰূপ ধাৰণ কৰে ।,Assamese
3,১৯৬৩ চনৰ নাজিৰা অধিৱেশনৰ অসম সাহিত্য সভাৰ সভাপ...,Assamese
4,"সৰু লৰা - ছোৱালী , ১২ বছৰৰ তলৰ হ ' লে , তুলসীৰ...",Assamese
...,...,...
943971,( maktubaat amaam rabanio; jeej 2 sas 225 ),Urdu
943972,karaachi mein korangi aur old stee airiya mein...,Urdu
943973,elawah azein iss say qabal bhee rawaan saal ag...,Urdu
943974,kaazimi saahib kay kees say inn kuu pehchaan m...,Urdu


In [53]:
# Calculate size in bytes → MB / GB
df_size_bytes = phase2.memory_usage(deep=True).sum()
df_size_mb = df_size_bytes / (1024 ** 2)
df_size_gb = df_size_bytes / (1024 ** 3)

print(f"\nTotal samples: {len(phase2):,}")
print(f"Languages included: {phase2['label'].nunique()}")
print(f"DataFrame size: {df_size_mb:.2f} MB ({df_size_gb:.3f} GB)")


Total samples: 943,976
Languages included: 22
DataFrame size: 252.25 MB (0.246 GB)


In [54]:
phase2.to_csv('phase2.csv', index=False)